In [ ]:
import requests

BASE_URL = "" # Enter your URL
API_KEY = "" # Enter your token

HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

LLM_PROVIDERS_ENDPOINT = "/v3/llm-gateway/providers"
CONFIG_OPTIONS_ENDPOINT = "/v3/evaluators/get-config-options"
EVALUATORS_ENDPOINT = "/v3/evaluators"
EVALUATOR_RULES_ENDPOINT = "/v3/evaluator-rules"

# `GET /v3/llm-gateway/providers`

Lists **LLM Gateway providers** configured for the current organization: each provider includes credentials and registered models (with UUIDs). Use this when wiring **`llm_as_a_judge`** evaluators so you can pick `model_id` and `credential_id` for `POST /v3/evaluators`.

## Query parameters

| Parameter | Description |
|-----------|-------------|
| `filter` | Optional. JSON string: `{"condition":"AND"\|"OR","rules":[...]}` (standard v3 list filter shape). |
| `search` | Optional. Free-text search. |
| `ordering` | Optional. Comma-separated fields; prefix `-` for descending. |
| `offset` | Optional. Default `0`. |
| `limit` | Optional. Page size. |

## Response

Standard **paginated** envelope: `kind`, `api_version`, and `data` with page fields plus `items`.

Each item typically includes (among others):

- `uuid` — provider row UUID  
- `provider` — provider name / key  
- `credentials` — list of `{ name, uuid, ... }`  
- `models` — list of `{ name, uuid, ... }`  
- `created_at`, `updated_at`, `created_by` (if present)


In [ ]:
llm_providers = requests.get(
    url=BASE_URL + LLM_PROVIDERS_ENDPOINT,
    headers=HEADERS
).json()["data"]["items"]

llm_providers

# `GET /v3/evaluators/get-config-options`

Returns **JSON Schema** (and display metadata) for **each active evaluator / enrichment type**, so clients can build a valid `enrichment_config` before calling `POST /v3/evaluators`.

## Request

```
GET /v3/evaluators/get-config-options
```

## Response

Standard success envelope: `data` contains a **`config_schemas`** array. Each element includes:

- `enrichment_name` — string enum value (e.g. `llm_as_a_judge`, `sentiment_analysis`)
- `enrichment_display_name` — human-readable label
- `enrichment_config` — JSON Schema for that type’s config object
- `preset_fields` — reserved for pre-filled / non-editable fields (often empty)


In [ ]:
evaluator_configs = requests.get(
    url=BASE_URL + CONFIG_OPTIONS_ENDPOINT,
    headers=HEADERS
).json()["data"]["config_schemas"]

evaluator_configs

# `POST /v3/evaluators`

Creates an **evaluator** at organization scope: it defines *what* to run (enrichment type + config), not *where* (no application binding). Bind an evaluator to an app with `POST /v3/evaluator-rules`.


## Request body

| Field | Required | Description |
|-------|----------|-------------|
| `name` | Yes | Unique name, 1–256 characters. |
| `enrichment_name` | Yes | Evaluator type (see table below). |
| `enrichment_config` | Depends on type | Omitted or `{}` for types that use empty config; required shape from `GET /v3/evaluators/get-config-options`. |

## Enrichment names and configs

Use the `/v3/get-config-options` endpoint mentioned above to fetch valid `enrichment_name` and `enrichment_config` combinations.

## Credential IDs

Use the `/v3/llm-gateway/providers` endpoint mentioned above to fetch valid `credential_id` values.

In [ ]:
# Example of a simple evaluator
safety_payload = {
    "name": "My Safety Evaluator 1234",
    "enrichment_name": "ftl_prompt_safety"
}

# Example of an evaluator with an external model provider
answer_relevance_payload = {
    "name": "Test Answer Relevance Evaluator 1",
    "enrichment_name": "answer_relevance_v2",
    "enrichment_config": {
        "model_id": "50dd002d-3d00-41f2-a0b0-159cc8d5d69b",
        "credential_id": "0b0ebc25-7f7b-46a1-8d2a-d568ac5b7f47"
    }
}

# Example of a custom LLM-as-a-Judge evaluator
custom_judge_payload = {
    "name": "My LLM as a Judge 1",
    "enrichment_name": "llm_as_a_judge",
    "enrichment_config": {
        "model_id": "50dd002d-3d00-41f2-a0b0-159cc8d5d69b",
        "credential_id": "0b0ebc25-7f7b-46a1-8d2a-d568ac5b7f47",
        "prompt_spec": {
            "prompt_template": [
                {
                    "role": "user",
                    "content": "Tell me if this prompt is good: {{input}}"
                }
            ],
            "output_fields": {
                "good_score": {
                    "type":"number",
                    "description": "How good is the prompt"
                }
            }
        }
    }
}

In [ ]:
PAYLOAD = custom_judge_payload # Customize the evaluator payload

# Create the evaluator
response = requests.post(
    url=BASE_URL + EVALUATORS_ENDPOINT,
    headers=HEADERS,
    json=PAYLOAD
)
response.json()

# `POST /v3/evaluator-rules`

Creates an **evaluator rule**: binds an existing evaluator to a **GenAI application**, maps span attributes to evaluator inputs, optionally filters spans, and optionally starts a **backfill**. New matching traces are evaluated after the rule exists.

If **`backfill`: true** and submitting the backfill job fails, the rule is **rolled back** and the API may respond with **503**.

## Request body

| Field | Required | Description |
|-------|----------|-------------|
| `name` | Yes | Rule name; must be **unique within the application**. |
| `application_id` | Yes | GenAI application UUID. |
| `evaluator_id` | Yes | Evaluator UUID from `GET`/`POST /v3/evaluators`. |
| `mapped_input_keys` | Yes | Object: keys match the evaluator’s **input schema**; values are **span attribute path strings** (e.g. Fiddler content keys). Example for sentiment: `{"text": "fiddler.contents.gen_ai.llm.output"}`. Use `POST /v3/evaluator-rules/map-input-keys` to list `attribute_names` and the input JSON schema. |
| `filters` | No | Limits which spans run the evaluator. Only **`SpanName`** and **`SpanType`** are allowed as `field` values (trace column names). Structure: `{ "condition": "AND" \| "OR", "rules": [ QueryRule \| nested condition ] }`. |
| `backfill` | No | Default `false`. If `true`, enqueues historical span evaluation. |
| `backfill_start_time` | No | ISO 8601 lower bound for backfill when `backfill` is true. |

### Operators in `filters`

Use **lowercase** string operators, e.g. `equal`, `in`, `greater`, `less_or_equal`. The query layer accepts documented aliases (e.g. `exact` → equal).

### `filters` example

```json
{
  "condition": "AND",
  "rules": [
    {
      "field": "SpanName",
      "operator": "equal",
      "value": "my_span_name"
    }
  ]
}
```

## Response

**200** — `data` contains the created rule: `id`, `name`, `evaluator_id`, `evaluator_name`, `evaluator_type`, `mapped_input_keys`, `filters`, `outputs`, `enabled`, timestamps, etc.

**409** — duplicate rule name in the same application.

In [ ]:
EVALUTOR_RULE_PAYLOAD = {
    "name": "My Evaluator Rule",
    "application_id": "5777262e-357f-4c3c-a7c8-d0e959ada60c",
    "evaluator_id": "59255db8-ac2c-4ea4-9f04-38f3deca2737",
    "filters": {
        "condition": "AND",
        "rules": [
            {
                "field": "SpanType",
                "operator": "IN",
                "value": [
                    "llm"
                ]
            },
            {
                "field": "SpanName",
                "operator": "IN",
                "value": [
                    "my_span_name"
                ]
            }
        ]
    },
    "mapped_input_keys": {
        "input": "fiddler.contents.gen_ai.llm.input.user",
        # "retrieved_documents": "fiddler.contents.gen_ai.llm.input.system",
        # "rag_response": "fiddler.contents.gen_ai.llm.output"
    },
    "backfill": False
}

In [ ]:
# Create the evaluator rule
response = requests.post(
    url=BASE_URL + EVALUATOR_RULES_ENDPOINT,
    headers=HEADERS,
    json=EVALUTOR_RULE_PAYLOAD
)
response.json()